## Analyzing New York's Yellow taxi dataset

- The datasets can be accessed from [here](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- Time period: `January 2022 - December 2025`
- visit [`here`](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) to know what each column means
- Dataset Characteristics <br>
 -> File Format: Parquet  
 -> Dataset type: Tabular  
 -> Number of columns: 19. 
 -> Time Period: 4 years (January - December)
-> 2022:
-> 2023:
-> 2024:
-> 2025: 

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet('../raw/2022/yellow_tripdata_2022-01.parquet')

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df.info()

## Columns and Defenitions

* `VendorID`: A code indicating the TPEP provider that provided the record. <br>
-> 1 = Creative Mobile Technologies, LLC  
-> 2 = Curb Mobility, LLC  
-> 6 = Myle Technologies Inc  
-> 7 = Helix  

* `tpep_pickup_datetime`: The date and time when the meter was engaged.
* `tpep_dropoff_datetime`: The date and time when the meter was disengaged.
* `passenger_count`: The number of passengers in the vehicle.
* `trip_distance`: The elapsed trip distance in miles reported by the taximeter.
* `RatecodeID`: The final rate code in effect at the end of the trip. <br>
-> 1 = Standard rate  
-> 2 = JFK  
-> 3 = Newark  
-> 4 = Nassau or Westchester  
-> 5 = Negotiated fare  
-> 6 = Group ride  
-> 99 = Null/unknown  
* `store_and_fwd_flag`: This flag indicates whether the trip record was held in vehicle memory before
sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server. <br>
-> Y = store and forward trip  
-> N = not a store and forward trip  
* `PULocationID`: TLC Taxi Zone in which the taximeter was engaged.
* `DOLocationID`: TLC Taxi Zone in which the taximeter was disengaged.
* `payment_type`: A numeric code signifying how the passenger paid for the trip. <br>
-> 0 = Flex Fare trip  
-> 1 = Credit card  
-> 2 = Cash  
-> 3 = No charge  
-> 4 = Dispute  
-> 5 = Unknown  
-> 6 = Voided trip  
* `fare_amount`: The time-and-distance fare calculated by the meter.
* `extra`: Miscellaneous extras and surcharges.
* `mta_tax`: Tax that is automatically triggered based on the metered rate in use.
* `tip_amount`: Tip amount – This field is automatically populated for credit card tips. Cash tips are not included.
* `tolls_amount`: Total amount of all tolls paid in trip
* `improvement_surcharge`: Improvement surcharge assessed trips at the flag drop
* `total_amount`: The total amount charged to passengers. Does not include cash tips. (Total amount is the sum of `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, and `improvement_surcharge`)
* `congestion_surcharge`: Total amount collected in trip for NYS congestion surcharge.
* `airport_fee`: For pick up only at LaGuardia and John F. Kennedy Airports
* `cbd_congestion_fee`: Per-trip charge for MTA's Congestion Relief Zone starting Jan. 5, 2025.

In [ ]:
df.head()

## Using DuckDB for quering through the data

In [ ]:
import duckdb

In [ ]:
df = duckdb.read_parquet('../raw/*/*.parquet')

In [ ]:
df.columns

In [ ]:
df = duckdb.sql("""
                SELECT * EXCLUDE (
                VendorID,
                store_and_fwd_flag
                )
                FROM df
                """)

In [ ]:
df

## Finding negative values in numeric fields

In [ ]:
duckdb.sql("""
            SELECT 
            COUNT (*) AS total_rows,
            COUNT (*) FILTER (WHERE trip_distance < 0) AS negative_distance,
            COUNT (*) FILTER (WHERE fare_amount < 0) AS negative_fare,
            COUNT (*) FILTER (WHERE passenger_count < 0) AS negative_passengers,
            COUNT (*) FILTER (WHERE extra < 0) AS negative_extra,
            COUNT (*) FILTER (WHERE mta_tax < 0) AS negative_mta_tax,
            COUNT (*) FILTER (WHERE tip_amount < 0) AS negative_tip_amount,
            COUNT (*) FILTER (WHERE tolls_amount < 0) AS negative_tolls_amount,
            COUNT (*) FILTER (WHERE improvement_surcharge < 0) AS negative_improvement_surcharge,
            COUNT (*) FILTER (WHERE total_amount < 0) AS negative_total_amount,
            COUNT (*) FILTER (WHERE congestion_surcharge < 0) AS negative_congestion_surcharge,
            COUNT (*) FILTER (WHERE airport_fee < 0) AS negative_airport_fee
            FROM df
""").df()

## Negative values

- Negative fare: `4214194`
- Negative extra: `1029414`
- Negative mta tax: `1958014`
- Negative tip amount: `8229`
- Negative tolls amount: `158379`
- Negative improvement surcharge: `2042089`
- Negative total amount: `2215654`
- Negative congestion surcharge: `1639288`
- Negative airport fee: `344818`

In [ ]:
duckdb.sql("""
            SELECT * 
            FROM df
            WHERE 
            fare_amount < 0 
            
            """)

## Finding average passengers

In [ ]:
duckdb.sql("""
            SELECT 
            AVG(trip_distance) AS average_trip_distance,
            MIN(trip_distance) AS minimum_trip_distance,
            MAX(trip_distance) AS maximum_trip_distance,
            PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY trip_distance) AS p25,
            PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY trip_distance) AS p50,
            PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY trip_distance) AS p75,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance) AS P99
            FROM df 
            """)

- The above analysis shows that 99 percent of the trips are under 20 miles.
- the maximum trip distance is 398608.62 (outlier)

In [ ]:
duckdb.sql("""
            SELECT
            *
            FROM df
            WHERE trip_distance > 100
            """)

## Extracting Trip Distance Time

In [ ]:
trip_tw = duckdb.sql("""
                SELECT 
                    trip_distance,
                    EXTRACT(EPOCH FROM (
                        tpep_dropoff_datetime - tpep_pickup_datetime
                        )) / 60 AS trip_duration_minutes,
                        fare_amount,
                        total_amount
                FROM df
                WHERE trip_distance > 20
                ORDER BY trip_distance DESC
                """).df()

trip_tw

Negative values are probably refunds 

In [ ]:
neg_fairs = duckdb.sql("""
                        SELECT 
                            total_amount AS neg_total_amount,
                            payment_type
                        FROM
                            df
                        WHERE 
                            total_amount < 0 
                        """).df()
neg_fairs

## References

- https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
- https://duckdb.org/docs/current/clients/python/overview